# PID from clusters to neural networks — a three-hour O2 tutorial
**Learn by watching a fit change, then changing the problem.** No prior ML experience is assumed.

| Time | Activity |
|---|---|
| 0–10 min | Setup and the two-dimensional PID plane |
| 10–30 min | k-means with k=3: assign points, move centres |
| 30–50 min | Gaussian mixtures: watch EM iterations |
| 50–65 min | Exercise 1: momentum range and overlapping species |
| 65–75 min | Break |
| 75–95 min | Real skim: AP mother geometry, daughter roles and feature shapes |
| 95–130 min | Seven-component GMM, ReLU MLP and feature transformer |
| 130–145 min | Exercise 2: size versus architecture |
| 145–160 min | Export all models to ONNX FP32 and FP16 |
| 160–180 min | O2Physics inference and Exercise 3: benchmark CSV plots |

Run top to bottom; optional AP derivations and extended scans can be revisited after the session. The worked exercises have functioning defaults. Preinstall `numpy pandas matplotlib scipy scikit-learn torch onnx onnxruntime uproot jupyterlab nbformat nbclient`; prepare the ROOT skim and the O2 environment before the session. Python CPU execution is supported; GPU benchmarking needs a GPU-enabled O2 ONNX Runtime. No O2Physics build is performed here.

**Two distinct data sources:** the opening common-pion nσ lesson uses an explicit **detector-response toy**, not measured ALICE PID. The supplied skim stores assigned-hypothesis TOF nσ, not fixed-pion TOF nσ or the path/time/resolution needed to reconstruct it. Adding a label-dependent expected time with an assumed path would build separation into the input. We do not use that shortcut. The second half trains on the **real ROOT skim**, using measured TPC signal, momentum and AP variables. If genuine fixed-pion nσ data become available, the opening loader accepts a CSV with the documented columns.

In [1]:
from pathlib import Path
import copy, json, random, platform, hashlib, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse, Patch
from scipy.special import logsumexp
from scipy.optimize import linear_sum_assignment
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.mixture import GaussianMixture
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    recall_score,
    precision_score,
    ConfusionMatrixDisplay,
)
from IPython.display import display
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader


In [2]:
SEED = 42


In [3]:
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(min(4, torch.get_num_threads()))
DEVICE = torch.device("cpu")  # change to 'cuda' if prepared; not needed for learning
BASE = Path.cwd()
SPECIES = {2: ("π", "#0072B2"), 3: ("K", "#D55E00"), 4: ("p", "#009E73")}
species_name = lambda k: SPECIES[int(k)][0]
species_color = lambda k: SPECIES[int(k)][1]
plt.rcParams.update({"figure.dpi": 100, "axes.grid": True, "grid.alpha": 0.2})


In [4]:
INTRO_CSV = None  # optional genuine data: p, nsigTPC_pi, nsigTOF_pi, pid (2,3,4)
INTRO_RANGE = (0.8, 1.0)
DATA_SOURCE = "root"  # for the AP section; 'synthetic' is a fallback
# Git sample: 20,000 selected tracks, six branches, 0.8 <= p < 1.0 GeV/c.
# For other real-data momentum ranges, use tutorial_tpcskimv0.root (0.4 <= p < 5).
ROOT_PATH = str(BASE / "tutorial_tpcskimv0_git.root")
TREE_NAME = "O2tpcskimv0tree"
P_MIN, P_MAX = 0.8, 1.0  # separate real-data AP momentum selection
MAX_ROWS = 20_000


In [5]:
# Python/NumPy translation of the ALICE O2 ALEPH Bethe–Bloch response.
# Original headers: CERN/ALICE O2, GPL v3; see their copyright/license notices.
BB_PARAMS = (0.1885748433, 4.362178362, 0.005801536533, 2.347828417, 0.9323431977)
MIP = 50.0
CHARGE_FACTOR = 2.3
PION_MASS = 0.1395704  # O2 CommonConstants/PhysicsConstants.h: MassPionCharged


In [6]:
def bethe_bloch_aleph(bg, params=BB_PARAMS):
    bg = np.asarray(bg, dtype=np.float64)
    p1, p2, p3, p4, p5 = params
    with np.errstate(divide="ignore", invalid="ignore", over="ignore"):
        beta = bg / np.sqrt(1.0 + bg * bg)
        aa = np.power(beta, p4)
        bb = np.log(p3 + np.power(1.0 / bg, p5))
        result = (p2 - aa - bb) * p1 / aa
    return np.where(bg > 0, result, np.nan)


def expected_pion_signal(momentum):
    return (
        MIP
        * bethe_bloch_aleph(np.asarray(momentum) / PION_MASS)
        * abs(1.0) ** CHARGE_FACTOR
    )


## 1. A familiar PID plane
For a **fixed pion reference**, $n\sigma_D^\pi=(s_D-s_D^{expected}(\pi,p))/\sigma_D$, where D is TPC or TOF. Pions tend to lie around (0,0); other masses produce different expected signals. The reference hypothesis must be the same for all tracks. A **feature** is an input number; a **label** is the species used to check a prediction, not an input.

The toy generates measured TPC signal around an approximate Bethe–Bloch expectation, and measured TOF time around $t=L\sqrt{1+(m/p)^2}/c$, with Gaussian detector errors. It then computes pion-reference nσ from those synthetic measurements. Mass labels are used to **generate** the toy only; the clustering receives just two numbers. The constants are illustrative, not an ALICE calibration. The sample has equal numbers of π, K, p so class imbalance does not obscure the first lesson.

At fixed momentum, the three populations can be compact and separable. Broadening momentum mixes different mean responses into each population; going to high momentum reduces TOF mass separation. Neither a wide interval nor a larger model guarantees a particular outcome—look at the plots and measure it.

In [7]:
MASS_BY_PID = {2: PION_MASS, 3: 0.493677, 4: 0.9382720813}


In [ ]:
def make_pid_toy(n=90_000):
    rng = np.random.default_rng(SEED)
    pid = rng.choice([2, 3, 4], n)
    p = rng.uniform(0.4, 5.0, n)
    m = np.array([MASS_BY_PID[int(v)] for v in pid])
    expected_tpc = MIP * bethe_bloch_aleph(p / m)
    measured_tpc = expected_tpc * (1 + rng.normal(0, 0.06, n))
    pion_tpc = expected_pion_signal(p)
    L, c, sigma_t = 380.0, 29.9792458, 0.080  # cm, cm/ns, ns
    measured_tof = L / c * np.sqrt(1 + (m / p) ** 2) + rng.normal(0, sigma_t, n)
    pion_tof = L / c * np.sqrt(1 + (PION_MASS / p) ** 2)
    return pd.DataFrame(
        {
            "p": p,
            "pid": pid,
            "nsigTPC_pi": (measured_tpc - pion_tpc) / (0.06 * pion_tpc),
            "nsigTOF_pi": (measured_tof - pion_tof) / sigma_t,
        }
    )


In [ ]:
intro_data = make_pid_toy() if INTRO_CSV is None else pd.read_csv(INTRO_CSV)
INTRO_SOURCE = (
    "Detector-response toy" if INTRO_CSV is None else "Supplied fixed-pion PID data"
)
needed = ["p", "pid", "nsigTPC_pi", "nsigTOF_pi"]
assert set(needed).issubset(intro_data.columns)
intro_data = intro_data.loc[
    np.isfinite(intro_data[needed]).all(axis=1) & intro_data.pid.isin(SPECIES)
].copy()
PID_FEATURES = ["nsigTPC_pi", "nsigTOF_pi"]


In [ ]:
def pid_slice(interval):
    frame = intro_data.loc[intro_data.p.between(*interval, inclusive="left")].copy()
    if (
        set(frame.pid.unique()) != set(SPECIES)
        or frame.groupby("pid").size().min() < 30
    ):
        raise ValueError(
            "Need all three species with >=30 rows each; widen the interval."
        )
    return frame.sample(min(6000, len(frame)), random_state=SEED).reset_index(drop=True)


def show_pid(ax, frame, title):
    for pid, g in frame.groupby("pid"):
        ax.scatter(
            g.nsigTPC_pi,
            g.nsigTOF_pi,
            s=5,
            alpha=0.35,
            color=species_color(pid),
            label=species_name(pid),
        )
    ax.set(
        xlabel="TPC nσ (pion reference)", ylabel="TOF nσ (pion reference)", title=title
    )
    ax.legend(markerscale=2)


In [ ]:
intro = pid_slice(INTRO_RANGE)
fig, ax = plt.subplots(figsize=(7, 5))
show_pid(ax, intro, f"{INTRO_SOURCE}: {INTRO_RANGE} GeV/c")
plt.tight_layout()
plt.show()


In [ ]:
# Keep fit, component-naming and evaluation rows separate.
itr, ihold = train_test_split(
    np.arange(len(intro)), test_size=0.4, stratify=intro.pid, random_state=SEED
)
ical, iev = train_test_split(
    ihold, test_size=0.5, stratify=intro.pid.iloc[ihold], random_state=SEED
)
pid_scaler = StandardScaler().fit(intro[PID_FEATURES].iloc[itr])
Z = pid_scaler.transform(intro[PID_FEATURES])
Zfit = Z[itr]
print("Fit / name / evaluate:", len(itr), len(ical), len(iev))


## 2. k-means: three moving centres (20 minutes)
Choose k=3 cluster centres. Repeat **assign → update**:
1. Assign each row to the nearest centre using squared Euclidean distance.
2. Move each centre to the mean of its assigned rows.

This reduces the within-cluster sum of squares, $J=\sum_i\|x_i-\mu_{c_i}\|^2$. Membership is **hard**: one point, one cluster. Centres are not species labels; colour below means cluster ID. There is no mass information in the fit. The dashed lines are the **Voronoi partition**: the nearest-centre rule under Euclidean distance always cuts space with straight perpendicular bisectors between centre pairs, whatever the data looks like.

We standardize each input with the **fit-set** mean and standard deviation so one coordinate's numerical range does not dominate. Centroids and plots are converted back to nσ units for display. Euclidean distance in the fit coordinates implies roughly spherical clusters there; elongated populations and unequal densities challenge k-means. Different initial centres can give different local optima.

In [ ]:
CLUSTER_COLORS = np.array(
    ["#E69F00", "#CC79A7", "#56B4E9"]
)  # Okabe-Ito, disjoint from the species colours above


In [ ]:
def initial_centres(X, k=3):
    # k-means++-style distance-based seeding, no labels.
    rng = np.random.default_rng(SEED)
    centres = [X[rng.integers(len(X))]]
    for _ in range(1, k):
        d = ((X[:, None] - np.array(centres)[None]) ** 2).sum(2).min(1)
        centres.append(X[rng.choice(len(X), p=d / d.sum())])
    return np.array(centres)


def nearest(X, centres):
    return ((X[:, None] - centres[None]) ** 2).sum(2).argmin(1)


def kmeans_steps(X, iterations=15):
    centres = initial_centres(X)
    history = []
    for step in range(iterations + 1):
        lab = nearest(X, centres)
        history.append(
            {
                "step": step,
                "centres": centres.copy(),
                "labels": lab.copy(),
                "objective": float(((X - centres[lab]) ** 2).sum()),
            }
        )
        if step == iterations:
            break
        centres = np.array(
            [X[lab == k].mean(0) if np.any(lab == k) else centres[k] for k in range(3)]
        )
    return history


In [ ]:
km_history = kmeans_steps(Zfit)


In [ ]:
from scipy.spatial import Voronoi


# The Voronoi partition of the fitted centres (nearest-centre rule): plain Euclidean distance in
# standardized space, so bisectors are straight there; an affine map keeps them straight in raw units too.
def draw_voronoi(ax, centres_std, to_raw, color="k"):
    vor = Voronoi(centres_std)
    far = 50 * np.ptp(centres_std, axis=0).max() + 50
    centre_of_mass = centres_std.mean(axis=0)
    for (p1, p2), (v1, v2) in zip(vor.ridge_points, vor.ridge_vertices):
        if v1 == -1 or v2 == -1:
            v = v1 if v2 == -1 else v2
            t = centres_std[p2] - centres_std[p1]
            t = t / np.linalg.norm(t)
            n = np.array([-t[1], t[0]])
            mid = centres_std[[p1, p2]].mean(axis=0)
            n = n * np.sign(np.dot(mid - centre_of_mass, n) + 1e-12)
            seg_std = np.array([vor.vertices[v], vor.vertices[v] + n * far])
        else:
            seg_std = vor.vertices[[v1, v2]]
        seg_raw = to_raw(seg_std)
        ax.plot(seg_raw[:, 0], seg_raw[:, 1], color=color, lw=1.4, ls="--", zorder=5)


In [ ]:
raw_fit = intro[PID_FEATURES].iloc[itr].to_numpy()
pad = 0.06
xlim = np.quantile(raw_fit[:, 0], [0, 1]) + np.array([-1, 1]) * pad * np.ptp(
    raw_fit[:, 0]
)
ylim = np.quantile(raw_fit[:, 1], [0, 1]) + np.array([-1, 1]) * pad * np.ptp(
    raw_fit[:, 1]
)
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for ax, step in zip(axes.flat, [0, 1, 2, 5, 15]):
    state = km_history[step]
    centres = pid_scaler.inverse_transform(state["centres"])
    ax.scatter(*raw_fit.T, c=CLUSTER_COLORS[state["labels"]], s=5, alpha=0.4)
    draw_voronoi(ax, state["centres"], pid_scaler.inverse_transform)
    ax.scatter(*centres.T, c="black", marker="X", s=110, edgecolor="white", zorder=6)
    ax.set(
        xlim=xlim,
        ylim=ylim,
        title=f"After {step} updates",
        xlabel="TPC nσπ",
        ylabel="TOF nσπ",
    )
axes[1, 2].plot(
    [s["step"] for s in km_history], [s["objective"] for s in km_history], "o-"
)
axes[1, 2].set(
    xlabel="Update",
    ylabel="Sum of squared distances",
    title="Objective: lower is better",
)
plt.tight_layout()
plt.show()


## 3. GMM: learn centres, widths and soft membership (20 minutes)
A Gaussian has a mean (centre) and covariance (widths and correlations). In two dimensions its equal-density contours are ellipses. A mixture describes a density
$$p(x)=\sum_{k=1}^{K}\pi_k\,\mathcal N(x;\mu_k,\Sigma_k),\qquad \sum_k\pi_k=1.$$
The mixture weights $\pi_k$ describe component populations. Components need not correspond one-to-one to species.

**Expectation–maximization (EM)** alternates:
- **E:** calculate responsibilities $r_{ik}=\pi_k\mathcal N(x_i;\mu_k,\Sigma_k)/p(x_i)$. A boundary track can be, for example, 0.6 in one component and 0.4 in another.
- **M:** let $N_k=\sum_i r_{ik}$; update $\pi_k=N_k/N$, $\mu_k=\sum_i r_{ik}x_i/N_k$, and the corresponding weighted covariance. Repeat.

EM maximizes log likelihood, rather than minimizing k-means distances. Full covariance learns tilted and unequal-width clouds. A small diagonal regularizer prevents a covariance from collapsing; several restarts help local optima. Neither algorithm knows the species labels. Here “iteration” is one E/M cycle, not a neural-network epoch.

**Reading the plot:** point colors blend the soft responsibilities. Colored ellipses mark Mahalanobis radius 2: they describe each Gaussian's width and correlation, enclosing about 86.5% of that component's probability in two dimensions. They are **not decision boundaries**, and a Gaussian has nonzero density outside its ellipse.

EM fits a density and soft memberships; it does not need a hard decision boundary. We therefore hide hard boundaries by default. Set `SHOW_GMM_BOUNDARIES = True` below to inspect them.

For a hard assignment, choose the component with the largest weighted density $\pi_k\mathcal N(x;\mu_k,\Sigma_k)$, equivalently the largest responsibility. Its log score, apart from a common constant, is

$$s_k(x)=\log\pi_k-\tfrac12\log|\Sigma_k|-\tfrac12(x-\mu_k)^T\Sigma_k^{-1}(x-\mu_k).$$

Thus the decision depends on **population weight, Gaussian width/correlation, and distance from the mean**. It is neither a fixed sigma cutoff nor a count of nearby observed points. A visible boundary between components i and j satisfies $s_i=s_j$ **and both scores are at least as large as every other component's score**. Unequal covariances give quadratic boundaries, which can be open or enclose other components.

In this default toy, the broad yellow component wins across much of the surrounding plane, including far from its own cloud. Its boundaries lie around the narrower pink and blue regions; there need not be a closed boundary around the yellow points. Far from all clouds, even the winning component may have extremely low absolute density: a hard assignment alone does not indicate that a point is plausible.

In [ ]:
def log_components(X, means, covs, weights):
    result = []
    dim = X.shape[1]
    for mean, cov, weight in zip(means, covs, weights):
        L = np.linalg.cholesky(cov)
        whiten = np.linalg.solve(L, (X - mean).T).T
        result.append(
            np.log(weight)
            - np.log(np.diag(L)).sum()
            - 0.5 * dim * np.log(2 * np.pi)
            - 0.5 * (whiten**2).sum(1)
        )
    return np.column_stack(result)


In [ ]:
def em_steps(X, iterations=30):
    means = initial_centres(X)
    covs = np.repeat(np.cov(X.T)[None], 3, axis=0)
    weights = np.ones(3) / 3
    history = []
    for step in range(iterations + 1):
        logs = log_components(X, means, covs, weights)
        norm = logsumexp(logs, axis=1)
        r = np.exp(logs - norm[:, None])
        history.append(
            dict(
                step=step,
                means=means.copy(),
                covs=covs.copy(),
                weights=weights.copy(),
                responsibilities=r.copy(),
                likelihood=float(norm.mean()),
            )
        )
        if step == iterations:
            break
        nk = r.sum(0).clip(1e-8)
        weights = nk / nk.sum()
        means = r.T @ X / nk[:, None]
        delta = X[:, None, :] - means[None, :, :]
        covs = np.einsum("nk,nki,nkj->kij", r, delta, delta) / nk[
            :, None, None
        ] + 1e-4 * np.eye(X.shape[1])
    return history


In [ ]:
def ellipse_patch(ax, mean, cov, color):
    eig, vec = np.linalg.eigh(cov)
    angle = np.degrees(np.arctan2(vec[1, 1], vec[0, 1]))
    ax.add_patch(
        Ellipse(
            mean,
            4 * np.sqrt(eig[1]),
            4 * np.sqrt(eig[0]),
            angle=angle,
            edgecolor=color,
            facecolor="none",
            lw=2,
        )
    )


In [ ]:
def draw_gmm_boundary(ax, means, covs, weights, xlim, ylim, color="k", n=250):
    """Draw equal-score curves only where the tied components dominate."""
    xs = np.linspace(*xlim, n)
    ys = np.linspace(*ylim, n)
    xx, yy = np.meshgrid(xs, ys)
    grid = np.column_stack([xx.ravel(), yy.ravel()])
    grid_std = pid_scaler.transform(grid)
    scores = log_components(grid_std, means, covs, weights)
    scores = scores.reshape(*xx.shape, len(weights))

    # Component IDs are categories: contouring their numeric IDs creates
    # artificial interpolated levels. Contour pairwise log-score differences.
    for first in range(len(weights)):
        for second in range(first + 1, len(weights)):
            difference = scores[..., first] - scores[..., second]
            other_components = [
                k for k in range(len(weights)) if k not in (first, second)
            ]
            if other_components:
                other_best = scores[..., other_components].max(axis=-1)
                pair_best = np.maximum(scores[..., first], scores[..., second])
                difference = np.ma.masked_where(other_best > pair_best, difference)
            if difference.min() < 0 < difference.max():
                ax.contour(
                    xx,
                    yy,
                    difference,
                    levels=[0],
                    colors=color,
                    linewidths=1.4,
                    linestyles="--",
                    zorder=5,
                )


In [ ]:
SHOW_GMM_BOUNDARIES = (
    False  # optional hard assignments; EM itself uses soft memberships
)

em_history = em_steps(Zfit)


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
from matplotlib.colors import to_rgb

rgb = np.array([to_rgb(c) for c in CLUSTER_COLORS])
for ax, step in zip(axes.flat, [0, 1, 3, 10, 30]):
    state = em_history[step]
    ax.scatter(*raw_fit.T, c=state["responsibilities"] @ rgb, s=5, alpha=0.4)
    if SHOW_GMM_BOUNDARIES:
        draw_gmm_boundary(
            ax, state["means"], state["covs"], state["weights"], xlim, ylim
        )
    means = pid_scaler.inverse_transform(state["means"])
    for k in range(3):
        covariance = state["covs"][k] * np.outer(pid_scaler.scale_, pid_scaler.scale_)
        ellipse_patch(ax, means[k], covariance, CLUSTER_COLORS[k])
    ax.scatter(*means.T, c="black", marker="X", s=70, zorder=6)
    ax.set(
        xlim=xlim,
        ylim=ylim,
        title=f"EM after {step} updates",
        xlabel="TPC nσπ",
        ylabel="TOF nσπ",
    )
axes[1, 2].plot(
    [s["step"] for s in em_history], [s["likelihood"] for s in em_history], "o-"
)
axes[1, 2].set(
    xlabel="EM update",
    ylabel="Mean log likelihood",
    title="Fit objective: higher is better",
)
plt.tight_layout()
plt.show()
print(
    "Colored ellipses: Mahalanobis radius 2 (~86.5% per Gaussian in 2D), not decision boundaries."
)


## Exercise 1 — change the momentum range (15 minutes)
Predict first, then compare `(0.8,1.0)`, `(1.5,2.0)`, `(3.0,5.0)` and `(0.4,5.0)` GeV/c. Change `INTRO_RANGE` and rerun the introduction to watch the trajectories; the comparison cell below runs all four ranges automatically.

Record each model's balanced accuracy and inspect the clouds. Does a larger range produce more statistics but less Gaussian shapes? At high momentum, is the failure caused by curved populations or genuinely overlapping measurements? A flexible model cannot perfectly recover information the detector does not provide.

The fit uses no labels. A separate naming subset associates clusters with species; an evaluation subset measures the result. We use a one-to-one assignment for this deliberately three-cluster, three-species lesson. **Balanced accuracy** is mean per-species recall, so a rare class cannot be hidden by ordinary accuracy. Adding momentum as a third input is an optional extension; it tells a model where it is on the response curves.

In [ ]:
from sklearn.cluster import KMeans

RANGES = [(0.8, 1.0), (1.5, 2.0), (3.0, 5.0), (0.4, 5.0)]


In [ ]:
def name_three(labels, y):
    counts = np.zeros((3, 3), int)
    np.add.at(counts, (labels, y), 1)
    r, c = linear_sum_assignment(-counts)
    mapping = np.empty(3, int)
    mapping[r] = c
    return mapping


In [ ]:
range_results = []
fig, axes = plt.subplots(1, len(RANGES), figsize=(17, 4))
for ax, interval in zip(axes, RANGES):
    frame = pid_slice(interval)
    show_pid(ax, frame, str(interval) + " GeV/c")
    yy = frame.pid.map({2: 0, 3: 1, 4: 2}).to_numpy()
    a, b = train_test_split(
        np.arange(len(frame)), test_size=0.4, stratify=yy, random_state=SEED
    )
    c, d = train_test_split(b, test_size=0.5, stratify=yy[b], random_state=SEED)
    sc = StandardScaler().fit(frame[PID_FEATURES].iloc[a])
    zz = sc.transform(frame[PID_FEATURES])
    for name, model in [
        ("k-means", KMeans(3, n_init=10, random_state=SEED)),
        ("GMM", GaussianMixture(3, n_init=3, reg_covar=1e-4, random_state=SEED)),
    ]:
        model.fit(zz[a])
        mapping_demo = name_three(model.predict(zz[c]), yy[c])
        pred = mapping_demo[model.predict(zz[d])]
        range_results.append(
            dict(
                momentum=str(interval),
                model=name,
                balanced_accuracy=balanced_accuracy_score(yy[d], pred),
                rows=len(frame),
            )
        )
plt.tight_layout()
plt.show()
range_results = pd.DataFrame(range_results)
display(range_results)


**Break — 10 minutes.**

## 4. Real tracks: AP geometry and a harder feature space (20 minutes)
The real skim has one row per selected daughter, with `fPidIndex` assigned from its reconstructed decay. AP coordinates describe the **mother candidate**, not a separate pion or proton ellipse. A Λ supplies p and π⁻ at the **same raw (α,qT)**; an anti-Λ supplies p̄ and π⁺ at the same raw point on the mirrored mother band.

| Mother | Positive daughter | Negative daughter | Raw AP centre |
|---|---|---|---|
| K⁰ₛ | π⁺ | π⁻ | 0 |
| Λ | p | π⁻ | +(mp²−mπ²)/MΛ² |
| anti-Λ | π⁺ | p̄ | −(mp²−mπ²)/MΛ² |

Section 5's models use a charge-oriented version of this coordinate, $\tilde\alpha=\alpha\times\mathrm{sign}(\text{track charge})$, so a candidate's proton (or antiproton) role always lands on the same side regardless of which mirror half it started on. The plot below sticks to the ordinary, unoriented α -- the one usually drawn -- and shows the raw mother bands instead.

We use measured TPC signal divided by a common pion expectation, momentum, oriented α and qT as model inputs. We exclude assigned-hypothesis `fNSigTPC`, `fNSigTOF`, mass and PID index from the inputs. AP still closely encodes how the skim labels were selected, so high scores here are topology-label reproduction, not independent PID validation. A random row split can also share candidate/event information; real scientific validation needs event/run groups, which this simplified input does not supply.

In [ ]:
# PDG masses in GeV/c^2 (the pion mass PION_MASS comes from section 2).
M_PROTON, M_KAON, M_K0S, M_LAMBDA, M_OMEGA = (
    0.9382720813,
    0.493677,
    0.497611,
    1.115683,
    1.67245,
)


def two_body_pstar(M, m1, m2):
    return np.sqrt((M**2 - (m1 + m2) ** 2) * (M**2 - (m1 - m2) ** 2)) / (2 * M)


# hypothesis: parent mass, mass of THIS track, mass of the partner, species (fPidIndex) of this track
HYPOTHESES = {
    "K0S_pi": dict(
        name="K⁰ₛ → ππ   [π]",
        M=M_K0S,
        m_track=PION_MASS,
        m_other=PION_MASS,
        species=2,
        trend=True,
    ),
    "Lambda_p": dict(
        name="Λ → pπ   [p]", M=M_LAMBDA, m_track=M_PROTON, m_other=PION_MASS, species=4
    ),
    "Lambda_pi": dict(
        name="Λ → pπ   [π]", M=M_LAMBDA, m_track=PION_MASS, m_other=M_PROTON, species=2
    ),
    "Omega_K": dict(
        name="Ω → ΛK   [K]", M=M_OMEGA, m_track=M_KAON, m_other=M_LAMBDA, species=3
    ),
}


In [ ]:
# A self-contained sample: isotropic two-body decays, then small measurement errors.
# Each row has an independent generated parent (no paired-daughter leakage in this toy).


In [ ]:
def make_toy(n=120_000):
    rng = np.random.default_rng(SEED)
    role = rng.choice(list(HYPOTHESES), n, p=[0.60, 0.22, 0.08, 0.10])
    frames = []
    for key, h in HYPOTHESES.items():
        count = np.count_nonzero(role == key)
        M, m = h["M"], h["m_track"]
        ps = two_body_pstar(M, m, h["m_other"])
        P = rng.uniform(0.8, 8.0, count)
        beta = P / np.sqrt(P * P + M * M)
        gamma = 1 / np.sqrt(1 - beta * beta)
        u = rng.uniform(-1, 1, count)  # cos(theta*), NOT theta*
        qt = ps * np.sqrt(1 - u * u)
        longitudinal = gamma * (ps * u + beta * np.sqrt(m * m + ps * ps))
        p = np.hypot(longitudinal, qt)
        at = 2 * longitudinal / P - 1
        charge = rng.choice([-1, 1], count)
        frames.append(
            pd.DataFrame(
                {
                    "fTPCInnerParam": p * rng.normal(1, 0.003, count),
                    "fTPCSignal": expected_pion_signal(p)
                    * rng.normal({2: 1.0, 3: 1.12, 4: 1.8}[h["species"]], 0.10, count),
                    "fPidIndex": h["species"],
                    "fAlphaV0": charge * (at + rng.normal(0, 0.002, count)),
                    "fQtV0": qt + rng.normal(0, 0.001, count),
                    "fSigned1Pt": charge,
                }
            )
        )
    return pd.concat(frames, ignore_index=True)


In [ ]:
BRANCHES = [
    "fTPCInnerParam",
    "fTPCSignal",
    "fPidIndex",
    "fAlphaV0",
    "fQtV0",
    "fSigned1Pt",
]


In [ ]:
def select(frame):
    a = frame.fAlphaV0 * np.sign(frame.fSigned1Pt)
    mask = (
        np.isfinite(frame[BRANCHES]).all(axis=1)
        & frame.fTPCInnerParam.between(P_MIN, P_MAX, inclusive="left")
        & (frame.fTPCSignal > 0)
        & frame.fPidIndex.isin(SPECIES)
        & (frame.fSigned1Pt != 0)
        & (frame.fQtV0 >= 0)
        & (frame.fTPCInnerParam > frame.fQtV0)
        & (a > -0.99)
        & (a < 0.99)
    )
    return frame.loc[mask, BRANCHES].copy()


In [ ]:
if DATA_SOURCE == "synthetic":
    data = select(make_toy())
elif DATA_SOURCE == "root":
    import uproot

    ROOT_PATH = Path(ROOT_PATH).expanduser()
    if not ROOT_PATH.exists():
        raise FileNotFoundError(
            f"Supply the skim at {ROOT_PATH} or select synthetic mode."
        )
    cache_key = json.dumps(
        [
            str(ROOT_PATH.resolve()),
            ROOT_PATH.stat().st_size,
            ROOT_PATH.stat().st_mtime_ns,
            TREE_NAME,
            P_MIN,
            P_MAX,
            MAX_ROWS,
            SEED,
            "selection-v1",
        ]
    )
    cache = BASE / (
        "tutorial_sample_"
        + hashlib.sha256(cache_key.encode()).hexdigest()[:12]
        + ".npz"
    )
    # Bounded uniform reservoir: random priorities over ALL selected rows.
    if cache.exists():
        with np.load(cache) as saved:
            data = pd.DataFrame({key: saved[key] for key in BRANCHES})
    else:
        rng = np.random.default_rng(SEED)
        reservoir = pd.DataFrame()
        with uproot.open(ROOT_PATH) as root:
            matches = [
                k
                for k, v in root.classnames(recursive=True, cycle=False).items()
                if k == TREE_NAME or k.endswith("/" + TREE_NAME)
            ]
            if len(matches) != 1:
                raise ValueError(
                    f"Set TREE_NAME to one exact tree path; found {matches}"
                )
            for block in root[matches[0]].iterate(
                BRANCHES, step_size="50 MB", library="np"
            ):
                frame = select(pd.DataFrame(block))
                frame["_priority"] = rng.random(len(frame))
                reservoir = pd.concat([reservoir, frame]).nsmallest(
                    MAX_ROWS, "_priority"
                )
        data = reservoir.drop(columns="_priority")
        np.savez_compressed(cache, **{key: data[key].to_numpy() for key in BRANCHES})
    print("Prepared sample cache:", cache.name)
else:
    raise ValueError("DATA_SOURCE must be synthetic or root")


In [ ]:
data = data.sample(min(MAX_ROWS, len(data)), random_state=SEED).reset_index(drop=True)
data["fTPCSignalOverPionBB"] = data.fTPCSignal / expected_pion_signal(
    data.fTPCInnerParam
)
data["fAlphaTrack"] = data.fAlphaV0 * np.sign(data.fSigned1Pt)
print(DATA_SOURCE, "| retained tracks:", len(data))
display(data.groupby("fPidIndex").size().rename("tracks"))


In [ ]:
encoder = LabelEncoder()
y = encoder.fit_transform(data.fPidIndex)
CLASS_NAMES = [species_name(k) for k in encoder.classes_]
CLASS_COLORS = [species_color(k) for k in encoder.classes_]
N_CLASSES = len(encoder.classes_)
if N_CLASSES < 2 or np.bincount(y).min() < 10:
    raise ValueError(
        "Need at least two PID labels with >=10 rows each; widen the slice."
    )
idx_tr, idx_hold = train_test_split(
    np.arange(len(y)), test_size=0.4, stratify=y, random_state=SEED
)
idx_va, idx_te = train_test_split(
    idx_hold, test_size=0.5, stratify=y[idx_hold], random_state=SEED
)
ytr, yva, yte = y[idx_tr], y[idx_va], y[idx_te]
print(
    "Classes:",
    CLASS_NAMES,
    "| train / validation / test rows:",
    len(idx_tr),
    len(idx_va),
    len(idx_te),
)


### Mother bands, fitted to the data
One band per mother hypothesis, drawn in the raw $(\alpha,q_T)$ plane -- the ordinary way this plot is usually shown, with both daughter rows of a candidate landing on the same point. Each band's effective boost $\beta$ is **fitted to this momentum slice** (training rows only, robust least squares against that hypothesis's own tracks), rather than assumed. Shading is a $\pm1.5\%$ / $\pm4.5\%$ radial tolerance around the fitted curve, still not a detector resolution; a single $\beta$ is a compromise over the whole slice, since the true boost varies with momentum and angle.

In [ ]:
from scipy.optimize import least_squares

train = data.iloc[idx_tr]


# One effective beta per mother hypothesis, fit on training rows only (label-free w.r.t. validation/test).
def fit_beta(M, mp, mn, alpha, qt, beta0=0.95, window0=0.12, iters=5):
    ps = two_body_pstar(M, mp, mn)
    alpha0 = (mp * mp - mn * mn) / (M * M)
    beta, window = beta0, window0
    for _ in range(iters):
        rho = np.hypot((alpha - alpha0) * beta * M / (2 * ps), qt / ps)
        keep = np.abs(rho - 1) < window
        if keep.sum() < 30:
            break
        fit = least_squares(
            lambda b: np.hypot(
                (alpha[keep] - alpha0) * b[0] * M / (2 * ps), qt[keep] / ps
            )
            - 1,
            [beta],
            loss="soft_l1",
            f_scale=0.02,
        )
        beta = float(np.clip(fit.x[0], 0.05, 0.999))
        window = max(0.02, window * 0.7)
    return beta


In [ ]:
# (name, M, mass of the positive daughter, mass of the negative daughter, colour, species used to fit beta)
MOTHERS = [
    ("K⁰ₛ", M_K0S, PION_MASS, PION_MASS, "#0072B2", 2),
    ("Λ", M_LAMBDA, M_PROTON, PION_MASS, "#009E73", 4),
    ("anti-Λ", M_LAMBDA, PION_MASS, M_PROTON, "#984ea3", 4),
    ("Ω⁻ (cascade)", M_OMEGA, M_LAMBDA, M_KAON, "#D55E00", 3),
    ("anti-Ω⁺ (cascade)", M_OMEGA, M_KAON, M_LAMBDA, "#e6ab02", 3),
]


In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 5.8))
for pid, g in train.groupby("fPidIndex"):
    gg = g.sample(min(4000, len(g)), random_state=SEED)
    ax.scatter(
        gg.fAlphaV0,
        gg.fQtV0,
        s=3,
        alpha=0.25,
        color=species_color(pid),
        label=species_name(pid),
    )

ag, qg = np.meshgrid(np.linspace(-1, 1, 400), np.linspace(0, 0.26, 180))
fitted_beta = {}
alpha_tr, qt_tr, pid_tr = (
    train.fAlphaV0.to_numpy(),
    train.fQtV0.to_numpy(),
    train.fPidIndex.to_numpy(),
)
for name, M, mp, mn, color, species in MOTHERS:
    own = pid_tr == species
    beta = fit_beta(M, mp, mn, alpha_tr[own], qt_tr[own])
    fitted_beta[name] = beta
    ps = two_body_pstar(M, mp, mn)
    alpha0 = (mp * mp - mn * mn) / (M * M)
    rho = np.hypot((ag - alpha0) * beta * M / (2 * ps), qg / ps)
    ax.contourf(ag, qg, np.abs(rho - 1), levels=[0, 0.045], colors=[color], alpha=0.16)
    ax.contourf(ag, qg, np.abs(rho - 1), levels=[0, 0.015], colors=[color], alpha=0.38)
    theta = np.linspace(0, np.pi, 300)
    ax.plot(
        alpha0 + 2 * ps / (beta * M) * np.cos(theta),
        ps * np.sin(theta),
        color=color,
        lw=1.4,
        label=f"{name} (β≈{beta:.2f})",
    )
ax.set(
    xlim=(-1, 1),
    ylim=(0, 0.27),
    xlabel="α (raw, unchanged on both daughter rows)",
    ylabel="qT [GeV/c]",
    title=f"Mother hypotheses with beta fitted per hypothesis, {P_MIN} ≤ p < {P_MAX} GeV/c",
)
ax.legend(fontsize=7.5, ncol=2, loc="upper center")
plt.tight_layout()
plt.show()
print(
    "Fitted effective beta per hypothesis:",
    {k: round(v, 4) for k, v in fitted_beta.items()},
)


## 5. Same features, three models (35 minutes)
All three models train on the same four raw-ish inputs: `[momentum, TPC/pion expectation, oriented alpha, qT]` -- no ellipse unrolling, no decay-hypothesis residuals. The curved AP structure from section 4 is left exactly as measured; it is the models' job to find the boundary in it, not ours to straighten it out first. All three models always see the same chosen features, rows and scaling.

Split roles: 60% trains models and the scaler; the 20% validation split is subdivided into component-naming rows and model-selection rows; 20% test rows are used only after choices are fixed. Naming must not be scored on the same rows. We display validation metrics during exercises, then evaluate the frozen configurations once.

**Seven-component GMM:** seven Gaussian clouds can tile curves, but may spend components on abundant populations. Several components can name the same species. Class-balanced naming divides each component's class counts by total class counts, prioritizing mean recall; majority naming instead favours overall accuracy. The density fit is unsupervised; naming uses labels. These aggregated scores are not guaranteed calibrated probabilities.

In [ ]:
FEATURES = [
    "fTPCInnerParam",
    "fTPCSignalOverPionBB",
    "fAlphaTrack",
    "fQtV0",
]  # raw measurements only


In [ ]:
N_FEATURES = len(FEATURES)
Xraw = data[FEATURES].to_numpy(np.float32)
assert np.isfinite(Xraw).all()
scaler = StandardScaler().fit(Xraw[idx_tr])
Xtr, Xva, Xte = [
    scaler.transform(Xraw[i]).astype(np.float32) for i in [idx_tr, idx_va, idx_te]
]
name_rows, tune_rows = train_test_split(
    np.arange(len(yva)), test_size=0.5, stratify=yva, random_state=SEED + 1
)
Xname, yname = Xva[name_rows], yva[name_rows]
Xtune, ytune = Xva[tune_rows], yva[tune_rows]


In [ ]:
N_COMPONENTS = 7
MAPPING_RULE = "balanced"  # 'majority' is a useful imbalance exercise


In [ ]:
started = time.perf_counter()
gmm = GaussianMixture(
    N_COMPONENTS,
    covariance_type="full",
    n_init=5,
    max_iter=250,
    reg_covar=1e-4,
    random_state=SEED,
).fit(Xtr)
gmm_seconds = time.perf_counter() - started
# Soft responsibilities give every component evidence on naming rows.
counts = gmm.predict_proba(Xname).T @ np.eye(N_CLASSES)[yname]
scores = (
    counts / np.bincount(yname, minlength=N_CLASSES)[None, :]
    if MAPPING_RULE == "balanced"
    else counts
)
mapping = scores.argmax(1)
assignment = np.eye(N_CLASSES)[mapping]


def gmm_prob(X):
    return gmm.predict_proba(X) @ assignment


print(
    "GMM converged:",
    gmm.converged_,
    "iterations:",
    gmm.n_iter_,
    "fit seconds:",
    round(gmm_seconds, 2),
)
display(pd.DataFrame(counts, columns=CLASS_NAMES).round(1))
print(
    "Tuning balanced accuracy:",
    balanced_accuracy_score(ytune, gmm_prob(Xtune).argmax(1)),
)


### ReLU MLP: from measurements to a prediction
For one track, the input is four standardized measurements. For a batch of $B$ tracks, they form a matrix $X$ of shape $[B,4]$: rows are tracks and columns are features. Standardization uses training-set statistics only.

A dense layer computes $Z=XW^T+b$. With 64 hidden neurons, $W$ has shape $[64,4]$, the bias has shape $[64]$, and $Z$ is $[B,64]$. Each neuron learns a weighted combination of all four inputs. ReLU replaces negative entries with zero: $H=\max(0,Z)$. Stacking these transformations allows the network to learn nonlinear decision boundaries; without the nonlinearities, the stack would collapse to a single affine transformation.

The default MLP follows **[B,4] → Linear + ReLU → [B,64] → Linear + ReLU → [B,64] → Linear → [B,3]**. The final three numbers are **logits**, unrestricted scores for π, K and p in `CLASS_NAMES` order. They are not probabilities. Softmax converts them to $p_c=\exp(z_c)/\sum_j\exp(z_j)$; the largest score selects the predicted species. Confident predictions need not be calibrated or correct.

### How does the network learn?
For a track labeled $y$, cross entropy is $-\log p_y$. Predicting probability 0.8 for its reference species costs about 0.22; predicting 0.1 costs about 2.30. PyTorch's `CrossEntropyLoss` accepts **logits** and performs the numerically stable log-softmax internally.

1. **Forward:** compute logits for a batch and average the loss.
2. **Clear gradients:** `zero_grad()` removes gradients left from the preceding batch.
3. **Backward:** `loss.backward()` uses the chain rule to find how each parameter affects the loss. For one row, the derivative with respect to logit $z_c$ is $p_c-\mathbf{1}[c=y]$.
4. **Update:** AdamW uses these gradients to update weights and biases, with weight decay to discourage large weights. The learning rate controls update size.

An **epoch** visits the training rows once. Validation rows do not contribute gradients: they select the stopping epoch. If validation loss stops improving for `PATIENCE` epochs, we stop and restore the best weights. Falling training loss with rising validation loss suggests overfitting. The test set is reserved for evaluating the final choices.

In [ ]:
class MLP(nn.Module):
    """Map one feature vector per track to one logit per species."""

    def __init__(self, n_features, n_classes, width=64, depth=2):
        super().__init__()
        layers = []
        input_width = n_features
        for _ in range(depth):
            layers.append(nn.Linear(input_width, width))
            layers.append(nn.ReLU())
            input_width = width
        layers.append(nn.Linear(input_width, n_classes))
        self.net = nn.Sequential(*layers)

    def forward(self, features):
        return self.net(features)


### Feature transformer: what is a token here?
A language transformer represents words or word pieces as tokens. Here, **each feature of one track is a token**. With $F=4$ features and embedding width $D=16$, the input changes from $[B,4]$ to $[B,4,16]$:

$$T_{b,f,:}=X_{b,f}\,w_f+e_f.$$

Each feature has a learned vector $w_f$ and offset $e_f$, both of length 16. Thus momentum and qT retain distinct identities even when their standardized values happen to match. These feature-specific parameters play the role of identifying columns; we do not add sequence positions. Feature order must still match `FEATURES`.

### Attention: follow one matrix through the layer
Ignore the batch axis briefly and take one track's token matrix $T\in\mathbb{R}^{F\times D}$. This implementation first applies LayerNorm **across the D entries of each token**, giving $U$. LayerNorm has learned scale and bias and is separate from input standardization.

For one head with width $d=D/H$, learned projections produce

$$Q=UW_Q+b_Q,\quad K=UW_K+b_K,\quad V=UW_V+b_V,$$

where each projection matrix is $D\times d$ and $Q,K,V$ each have shape $F\times d$. A **query** describes what a token seeks, a **key** describes what it can match, and a **value** contains the information it can contribute. These are learned numerical representations, not predefined physical quantities.

| Step | Matrix shape (one head, one track) | Meaning |
|---|---|---|
| $S=QK^T/\sqrt d$ | $F\times F$ | Row i compares query i with every key j. Scaling limits growth in dot-product magnitude with head width. |
| $A_{ij}=\exp(S_{ij})/\sum_k\exp(S_{ik})$ | $F\times F$ | Softmax across columns: each row sums to one. |
| $C=AV$ | $F\times d$ | Output row i is $C_i=\sum_j A_{ij}V_j$, a weighted mixture of value rows. |

**Attention mixes feature tokens within one track. It never mixes tracks in the batch.** The attention matrix changes with the input, even after training has fixed the projection weights. There is no causal mask: all features are available together, so every feature may attend to every other feature and itself.

For the default $H=4$, $D=16$, each head has $d=4$. In code, Q, K and V have shape **[B,4 heads,4 features,4 channels]** and the attention weights have shape **[B,4 heads,4 queries,4 keys]**. Heads have separate learned projections. Concatenating their outputs restores [B,4,16]; a learned output projection mixes head channels.

### A small numerical attention example
The next cell isolates a single head with two tokens and two channels. We supply Q, K and V directly so that every multiplication is visible; these illustrative matrices are not fitted PID parameters. The first query prefers the first key, so its output is closer to the first value row. Try making both keys identical: each attention row becomes [0.5, 0.5].

In [ ]:
# A standalone arithmetic demonstration; it does not consume random numbers.
example_queries = torch.tensor([[1.0, 0.0], [0.0, 1.0]])
example_keys = torch.tensor([[1.0, 0.0], [0.0, 1.0]])
example_values = torch.tensor([[10.0, 0.0], [0.0, 20.0]])
example_scores = example_queries @ example_keys.T / 2**0.5
example_attention = torch.softmax(example_scores, dim=-1)
example_context = example_attention @ example_values
for label, matrix in [
    ("Q", example_queries),
    ("K", example_keys),
    ("V", example_values),
    ("Scaled scores", example_scores),
    ("Attention weights", example_attention),
    ("Output = attention @ V", example_context),
]:
    print(label)
    print(matrix.numpy().round(3))
# Approximately [[0.670, 0.330], [0.330, 0.670]] for attention,
# and [[6.698, 6.605], [3.302, 13.395]] for the output.
assert torch.allclose(example_attention.sum(dim=-1), torch.ones(2))


### From attention to a complete transformer block
Our **pre-normalized** block computes

$$T'=T+\mathrm{OutputProjection}(\mathrm{MultiHeadAttention}(\mathrm{LayerNorm}(T))),$$
$$T''=T'+\mathrm{FFN}(\mathrm{LayerNorm}(T')).$$

The additions are **residual connections**: each sublayer learns a correction while preserving a direct path for information and gradients. The feed-forward network (FFN) is Linear(D,2D) → ReLU → Linear(2D,D), applied separately to each token with shared weights. Attention exchanges information between tokens; the FFN transforms each token's channels. Both preserve [B,F,D].

After two blocks, mean pooling averages the four token vectors into [B,D]. A final linear layer returns [B,3] logits. Pooling happens **after** the learned interactions. This small classifier has no decoder, text generation, causal mask or dropout.

The code below spells out each reshape. The combined `qkv` projection is equivalent to three linear layers; packing them together saves work. `permute` changes axis order, while `reshape` splits or joins dimensions.

In [ ]:
class AttentionBlock(nn.Module):
    """Pre-normalized multi-head attention followed by a ReLU feed-forward block."""

    def __init__(self, width, heads):
        super().__init__()
        if heads <= 0 or width % heads != 0:
            raise ValueError("width must be divisible by a positive number of heads")
        self.heads = heads
        self.head_width = width // heads
        self.norm1 = nn.LayerNorm(width)
        self.norm2 = nn.LayerNorm(width)
        # One linear layer computes all three projections efficiently.
        self.qkv = nn.Linear(width, 3 * width)
        self.proj = nn.Linear(width, width)
        self.ff = nn.Sequential(
            nn.Linear(width, 2 * width), nn.ReLU(), nn.Linear(2 * width, width)
        )

    def forward(self, tokens):
        batch_size, n_tokens, width = tokens.shape
        normalized = self.norm1(tokens)  # [B, F, D]
        projections = self.qkv(normalized)  # [B, F, 3D]
        projections = projections.reshape(
            batch_size, n_tokens, 3, self.heads, self.head_width
        )  # [B, F, 3, H, d]
        projections = projections.permute(2, 0, 3, 1, 4)  # [3, B, H, F, d]
        queries, keys, values = projections.unbind(dim=0)

        scores = queries @ keys.transpose(-2, -1)  # [B, H, F, F]
        scores = scores / self.head_width**0.5
        attention = torch.softmax(scores, dim=-1)  # normalize each query row
        context = attention @ values  # [B, H, F, d]

        # Put tokens before heads, then concatenate the head vectors.
        context = context.transpose(1, 2).reshape(batch_size, n_tokens, width)
        tokens = tokens + self.proj(context)  # attention residual
        return tokens + self.ff(self.norm2(tokens))  # feed-forward residual


In [ ]:
class FeatureTransformer(nn.Module):
    def __init__(self, n_features, n_classes, width=16, depth=2, heads=4):
        super().__init__()
        self.value_weight = nn.Parameter(torch.randn(n_features, width) * 0.1)
        self.feature_bias = nn.Parameter(torch.randn(n_features, width) * 0.1)
        self.blocks = nn.Sequential(
            *[AttentionBlock(width, heads) for _ in range(depth)]
        )
        self.head = nn.Linear(width, n_classes)

    def forward(self, features):
        # Each scalar feature gets its own learned D-dimensional representation.
        tokens = features.unsqueeze(-1) * self.value_weight + self.feature_bias
        encoded_tokens = self.blocks(tokens)  # [B, F, D]
        track_embedding = encoded_tokens.mean(dim=1)  # [B, D]
        return self.head(track_embedding)  # [B, C], logits


def parameters(model):
    return sum(
        parameter.numel() for parameter in model.parameters() if parameter.requires_grad
    )


### Training settings and shared training loop
Both architectures use the same data splits, loss and training budget. Edit the settings below for Exercise 2, then rerun training and the validation plots. A larger network is not guaranteed to improve performance; compare validation quality, parameter count and fit time. Attention weights alone are not evidence of causal feature importance.

In [ ]:
MLP_CONFIG = dict(width=64, depth=2)
TRANSFORMER_CONFIG = dict(width=16, depth=2, heads=4)
EPOCHS = 45
BATCH_SIZE = 256
LEARNING_RATE = 1e-3
PATIENCE = 7
WEIGHT_DECAY = 1e-4


In [ ]:
class NNTrainer:
    def __init__(
        self,
        model,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        lr=LEARNING_RATE,
        patience=PATIENCE,
    ):
        self.model = model.to(DEVICE)
        self.epochs = epochs
        self.batch_size = batch_size
        self.lr = lr
        self.patience = patience

    def fit(self, X, y, Xval, yval):
        generator = torch.Generator().manual_seed(SEED)
        batches = DataLoader(
            TensorDataset(torch.from_numpy(X), torch.as_tensor(y, dtype=torch.long)),
            batch_size=self.batch_size,
            shuffle=True,
            generator=generator,
        )
        optimizer = torch.optim.AdamW(
            self.model.parameters(), lr=self.lr, weight_decay=WEIGHT_DECAY
        )
        loss_fn = nn.CrossEntropyLoss()
        best = float("inf")
        epochs_without_improvement = 0
        self.history = []
        start = time.perf_counter()
        for epoch in range(self.epochs):
            self.model.train()
            total = 0
            for batch_features, batch_labels in batches:
                batch_features, batch_labels = batch_features.to(
                    DEVICE
                ), batch_labels.to(DEVICE)
                optimizer.zero_grad()
                loss = loss_fn(self.model(batch_features), batch_labels)
                loss.backward()
                optimizer.step()
                total += loss.item() * len(batch_features)
            self.model.eval()
            validation_loss_sum = 0
            with torch.no_grad():
                for i in range(0, len(Xval), self.batch_size):
                    batch_features = torch.as_tensor(
                        Xval[i : i + self.batch_size], device=DEVICE
                    )
                    batch_labels = torch.as_tensor(
                        yval[i : i + self.batch_size], device=DEVICE, dtype=torch.long
                    )
                    validation_loss_sum += loss_fn(
                        self.model(batch_features), batch_labels
                    ).item() * len(batch_features)
            val = validation_loss_sum / len(Xval)
            self.history.append((total / len(X), val))
            if val < best:
                best = val
                state = copy.deepcopy(self.model.state_dict())
                self.best_epoch = epoch + 1
                epochs_without_improvement = 0
            else:
                epochs_without_improvement += 1
            if epochs_without_improvement >= self.patience:
                break
        self.model.load_state_dict(state)
        self.seconds = time.perf_counter() - start
        return self

    @torch.no_grad()
    def probabilities(self, X):
        self.model.eval()
        return np.concatenate(
            [
                torch.softmax(
                    self.model(
                        torch.as_tensor(X[i : i + self.batch_size], device=DEVICE)
                    ),
                    1,
                )
                .cpu()
                .numpy()
                for i in range(0, len(X), self.batch_size)
            ]
        )

    def predict(self, X):
        return self.probabilities(X).argmax(1)


### Train both networks
This is the cell to rerun after changing architecture settings. Training uses only `Xtr, ytr`; `Xtune, ytune` select the best epoch.

In [ ]:
trained = {}
for name, cls, config in [
    ("MLP", MLP, MLP_CONFIG),
    ("Transformer", FeatureTransformer, TRANSFORMER_CONFIG),
]:
    torch.manual_seed(SEED)
    trained[name] = NNTrainer(
        cls(N_FEATURES, N_CLASSES, **config),
        epochs=EPOCHS,
        lr=LEARNING_RATE,
        batch_size=BATCH_SIZE,
        patience=PATIENCE,
    ).fit(Xtr, ytr, Xtune, ytune)
    print(
        name,
        "parameters:",
        parameters(trained[name].model),
        "seconds:",
        round(trained[name].seconds, 1),
    )
mlp, transformer = trained["MLP"], trained["Transformer"]


### Inspect learning curves and validation metrics
The dotted line marks the restored epoch. Compare the validation curves as well as the final accuracy; training loss alone cannot select the better model.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, (name, trainer) in zip(axes, trained.items()):
    ax.plot(np.arange(1, len(trainer.history) + 1), trainer.history)
    ax.axvline(trainer.best_epoch, color="grey", ls=":")
    ax.legend(["Train", "Tuning validation", "Selected epoch"])
    ax.set(title=name, xlabel="Epoch", ylabel="Cross entropy")
plt.tight_layout()
plt.show()


In [ ]:
def quality(name, truth, pred, **extra):
    row = dict(
        model=name,
        accuracy=accuracy_score(truth, pred),
        balanced_accuracy=balanced_accuracy_score(truth, pred),
        **extra
    )
    for k, cls in enumerate(CLASS_NAMES):
        row["recall_" + cls] = recall_score(
            truth, pred, labels=np.arange(N_CLASSES), average=None, zero_division=0
        )[k]
        row["precision_" + cls] = precision_score(
            truth, pred, labels=np.arange(N_CLASSES), average=None, zero_division=0
        )[k]
    return row


In [ ]:
validation_results = pd.DataFrame(
    [quality("GMM", ytune, gmm_prob(Xtune).argmax(1), fit_seconds=gmm_seconds)]
    + [
        quality(
            name,
            ytune,
            t.predict(Xtune),
            parameters=parameters(t.model),
            fit_seconds=t.seconds,
            best_epoch=t.best_epoch,
        )
        for name, t in trained.items()
    ]
)
display(validation_results)


## Exercise 2 — size, architecture and overfitting (15 minutes)
Change only one factor first. Try MLP widths **16, 64, 128**, or depths **1, 2, 4**. Try transformer widths **8, 16, 32** and depths **1, 2** (width must be divisible by the number of heads). Keep the feature set, splits, loss and learning rate fixed. Rerun the training cell; compare parameter counts and validation results, not test scores.

The optional scan below compares a small and larger version of each architecture. Discuss: does more capacity improve validation loss, or only training loss? Does the larger model need more epochs? Are apparent architecture gains actually parameter-count or optimization-budget differences? Repeat promising comparisons with several seeds before claiming a winner. Early stopping restores the best validation epoch; it does not guarantee a model was sufficiently optimized.

**A useful counterexample is a valid result:** the transformer need not always outperform the MLP. Architecture affects how interactions are represented, while sample size, noise, feature engineering and optimization determine whether that helps. Low loss inside this split also does not prove generalization to a new run; a held-out momentum/run study is an optional extension.

In [ ]:
RUN_SIZE_SCAN = False  # enable during Exercise 2; baseline models above stay unchanged


In [ ]:
if RUN_SIZE_SCAN:
    size_rows = []
    for name, cls, config in [
        ("MLP-small", MLP, dict(width=16, depth=1)),
        ("MLP-large", MLP, dict(width=128, depth=3)),
        ("Transformer-small", FeatureTransformer, dict(width=8, depth=1, heads=2)),
        ("Transformer-large", FeatureTransformer, dict(width=32, depth=2, heads=4)),
    ]:
        torch.manual_seed(SEED)
        result = NNTrainer(
            cls(N_FEATURES, N_CLASSES, **config), epochs=EPOCHS, lr=LEARNING_RATE
        ).fit(Xtr, ytr, Xtune, ytune)
        size_rows.append(
            quality(
                name,
                ytune,
                result.predict(Xtune),
                parameters=parameters(result.model),
                fit_seconds=result.seconds,
                best_epoch=result.best_epoch,
            )
        )
    size_results = pd.DataFrame(size_rows)
    display(size_results)
    size_results.to_csv(BASE / "architecture_validation.csv", index=False)
else:
    print("Size scan is optional. Change configurations above for the short exercise.")


### Freeze choices, then evaluate once
The test set has not selected component names, architecture, stopping epochs or learning rates. Rows of the normalized confusion matrices are reference species; diagonal entries are recalls. Precision answers “of the tracks predicted as this species, what fraction carries that reference label?” The reference is topology-assigned, not independent truth.

In [ ]:
predictions = {
    "GMM": gmm_prob(Xte).argmax(1),
    **{name: t.predict(Xte) for name, t in trained.items()},
}
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, (name, pred) in zip(axes, predictions.items()):
    ConfusionMatrixDisplay.from_predictions(
        yte,
        pred,
        labels=np.arange(N_CLASSES),
        display_labels=CLASS_NAMES,
        normalize="true",
        ax=ax,
        colorbar=False,
    )
    ax.set_title(name)
plt.tight_layout()
plt.show()
test_results = pd.DataFrame(
    [quality(name, yte, pred) for name, pred in predictions.items()]
)
display(test_results)


### Where the errors are, and which sub-population they come from
A model can score well overall while still failing one physically meaningful **sub-population** it was never told about: `fPidIndex` cannot distinguish a K⁰ₛ-pion from a Λ-pion, since both carry the same species code. We can still group them for diagnosis, reusing the same fixed-reference mother bands drawn in section 4 -- evaluated once, directly at each raw $(\alpha,q_T)$, to say which band a track sits nearest to. This is **not** a model feature (it is computed only after training, purely to group the errors below) and **not** the per-track ellipse projection removed from section 4 -- just the same highlight curve, read off once per point.

In [ ]:
MOTHER_REFS = [
    ("K0s", M_K0S, PION_MASS, PION_MASS),
    ("Lambda", M_LAMBDA, M_PROTON, PION_MASS),
    ("antiLambda", M_LAMBDA, PION_MASS, M_PROTON),
    ("Omega", M_OMEGA, M_KAON, M_LAMBDA),
    ("antiOmega", M_OMEGA, M_LAMBDA, M_KAON),
]


def nearest_mother(alpha, qt, beta_ref=0.95):
    # Same fixed-beta approximation as the highlighted bands in section 4: a diagnostic label only.
    rhos = []
    for _, M, mp, mn in MOTHER_REFS:
        ps = two_body_pstar(M, mp, mn)
        alpha0 = (mp * mp - mn * mn) / (M * M)
        rhos.append(np.hypot((alpha - alpha0) * beta_ref * M / (2 * ps), qt / ps))
    idx = np.argmin(np.abs(np.column_stack(rhos) - 1), axis=1)
    return np.array([MOTHER_REFS[i][0] for i in idx])


def subpopulation(pid, mother):
    if pid == 2:
        return "π on the K⁰ₛ ellipse" if mother == "K0s" else "π on the Λ ellipse"
    if pid == 3:
        return "K on the Ω ellipse"
    return "p"


test_frame = data.iloc[idx_te].reset_index(drop=True)
nearest_te = nearest_mother(test_frame.fAlphaV0.to_numpy(), test_frame.fQtV0.to_numpy())
subpop_te = np.array(
    [subpopulation(pid, m) for pid, m in zip(test_frame.fPidIndex, nearest_te)]
)

fig, axes = plt.subplots(
    1, len(predictions), figsize=(5.3 * len(predictions), 4.6), sharey=True
)
for ax, (name, pred) in zip(axes, predictions.items()):
    wrong = pred != yte
    ax.scatter(test_frame.fAlphaV0, test_frame.fQtV0, s=4, alpha=0.2, color="lightgrey")
    ax.scatter(
        test_frame.fAlphaV0[wrong],
        test_frame.fQtV0[wrong],
        s=22,
        color="crimson",
        marker="x",
        linewidths=1.3,
        label="misclassified",
    )
    ax.set(xlabel=r"$\alpha$", title=f"{name}: {wrong.sum()} / {len(yte)} errors")
axes[0].set_ylabel(r"$q_T$ [GeV/c]")
axes[0].legend(loc="lower center", fontsize=8)
plt.tight_layout()
plt.show()

rows = []
for pop in ["π on the K⁰ₛ ellipse", "π on the Λ ellipse", "K on the Ω ellipse", "p"]:
    mask = subpop_te == pop
    row = {"sub-population": pop, "tracks": int(mask.sum())}
    for name, pred in predictions.items():
        row[name] = float(np.mean(pred[mask] == yte[mask])) if mask.sum() else np.nan
    rows.append(row)
display(pd.DataFrame(rows).set_index("sub-population"))


## 6. ONNX: freeze the computation for O2Physics (15 minutes)
We export **GMM, MLP and transformer**, each as FP32 and FP16 variants. ONNX stores weights and a computation graph, so O2 evaluates the model without Python or training. Batch size is dynamic.

**Input contract:** `features`, `[batch,N_FEATURES]`, engineered but **unstandardized** features in exactly `FEATURES` order. Scaling is inside the graph. FP32 models require float32 input; FP16 variants require float16 input. **Output:** `probabilities`, `[batch,N_CLASSES]`, in `encoder.classes_` order. The GMM output aggregates component posteriors by the saved naming rule.

FP16 has fewer significant digits and a smaller range; it can reduce memory and GPU cost, but may alter scores, class decisions or numerical stability. The conversion keeps sensitive Gaussian quadratic terms, softmax, layer normalization, subtraction and division in FP32, so the FP16 files are explicitly **mixed-precision graphs with FP16 interfaces**. The metadata records this; “FP16” does not mean every operation runs in half precision. No speedup is assumed.

We validate graph structure, multiple batch sizes, finite normalized scores and numerical agreement. FP16 checks use a CPU runtime if it supports the graph; CPU implementations may promote operations internally. This does not validate GPU kernels. The final O2 exercise must compare GPU outputs too.

In [ ]:
import onnx, onnxruntime as ort
from onnxruntime.transformers.float16 import convert_float_to_float16
from onnxruntime.transformers.onnx_model import OnnxModel


In [ ]:
class ProbabilityModel(nn.Module):
    """Raw feature vector in, class probabilities out. The standardisation is part of the graph."""

    def __init__(self, core, scaler, softmax=True):
        super().__init__()
        self.core, self.softmax = core, softmax
        self.register_buffer("mean", torch.tensor(scaler.mean_, dtype=torch.float32))
        self.register_buffer("scale", torch.tensor(scaler.scale_, dtype=torch.float32))

    def forward(self, x):
        out = self.core((x - self.mean) / self.scale)
        return torch.softmax(out, dim=1) if self.softmax else out


In [ ]:
class GMMPosterior(nn.Module):
    """Class posteriors of the fitted mixture: Gaussian log densities via Cholesky factors, softmax over
    components, then the component -> class mapping (a 0/1 matrix) found in section 5.2.
    """

    def __init__(self, gmm, mapping, n_classes):
        super().__init__()
        cov = (
            gmm.covariances_
            if gmm.covariance_type == "full"
            else np.array([np.diag(v) for v in gmm.covariances_])
        )
        L = torch.linalg.cholesky(torch.tensor(cov, dtype=torch.float64))
        K, d = gmm.means_.shape
        self.register_buffer("mu", torch.tensor(gmm.means_, dtype=torch.float32))
        self.register_buffer(
            "Linv_T", torch.linalg.inv(L).transpose(1, 2).float()
        )  # (K, d, d)
        self.register_buffer(
            "const",
            (
                torch.tensor(gmm.weights_, dtype=torch.float64).log()
                - torch.log(torch.diagonal(L, dim1=1, dim2=2)).sum(1)
                - 0.5 * d * np.log(2 * np.pi)
            ).float(),
        )
        self.register_buffer(
            "assign",
            torch.nn.functional.one_hot(torch.as_tensor(mapping), n_classes).float(),
        )  # (K, C)

    def forward(self, x):
        diff = (x[:, None, :] - self.mu).permute(1, 0, 2)  # (K, N, d)
        z = torch.matmul(diff, self.Linv_T).permute(
            1, 0, 2
        )  # (N, K, d): whitened residuals
        log_comp = self.const - 0.5 * (z * z).sum(-1)  # (N, K)
        return torch.softmax(log_comp, dim=1) @ self.assign  # (N, C)


In [ ]:
ONNX_DIR = BASE / "onnx_tutorial_v3"
ONNX_DIR.mkdir(exist_ok=True)


In [ ]:
wrappers = {
    "gmm": ProbabilityModel(
        GMMPosterior(gmm, mapping, N_CLASSES), scaler, softmax=False
    ).eval(),
    "mlp": ProbabilityModel(copy.deepcopy(mlp.model).cpu().eval(), scaler).eval(),
    "transformer": ProbabilityModel(
        copy.deepcopy(transformer.model).cpu().eval(), scaler
    ).eval(),
}
raw_test = np.ascontiguousarray(Xraw[idx_te], dtype=np.float32)
EXPORT_ROWS = min(1024, len(raw_test))
np.savetxt(ONNX_DIR / "features.txt", raw_test[:EXPORT_ROWS], fmt="%.9g")
np.savetxt(ONNX_DIR / "labels.txt", yte[:EXPORT_ROWS], fmt="%d")


In [ ]:
export_rows = []
variants = []
for name, wrapper in wrappers.items():
    path32 = ONNX_DIR / f"{name}_fp32.onnx"
    torch.onnx.export(
        wrapper,
        torch.zeros(2, N_FEATURES),
        str(path32),
        input_names=["features"],
        output_names=["probabilities"],
        dynamic_axes={"features": {0: "batch"}, "probabilities": {0: "batch"}},
        opset_version=17,
        dynamo=False,
    )
    graph = onnx.load(path32)
    onnx.checker.check_model(graph)
    # Gaussian quadratic terms can overflow half even when final probabilities are finite.
    blocks = ["Softmax", "LayerNormalization", "Sub", "Div"] + (
        ["Mul", "ReduceSum"] if name == "gmm" else []
    )
    half_graph = convert_float_to_float16(
        copy.deepcopy(graph), keep_io_types=False, op_block_list=blocks
    )
    # Conversion inserts Cast nodes: restore producer-before-consumer order.
    converted = OnnxModel(half_graph)
    converted.topological_sort()
    half_graph = converted.model
    path16 = ONNX_DIR / f"{name}_fp16.onnx"
    onnx.save(half_graph, path16)
    with torch.no_grad():
        reference = wrapper(torch.from_numpy(raw_test)).numpy()
    np.savetxt(
        ONNX_DIR / f"{name}_fp32_reference.txt", reference[:EXPORT_ROWS], fmt="%.9g"
    )
    for precision, path, dtype in [
        ("fp32", path32, np.float32),
        ("fp16", path16, np.float16),
    ]:
        model_graph = onnx.load(path)
        onnx.checker.check_model(model_graph)
        expected = (
            onnx.TensorProto.FLOAT if precision == "fp32" else onnx.TensorProto.FLOAT16
        )
        assert model_graph.graph.input[0].type.tensor_type.elem_type == expected
        assert model_graph.graph.output[0].type.tensor_type.elem_type == expected
        options = ort.SessionOptions()
        options.intra_op_num_threads = 2
        session = ort.InferenceSession(
            str(path), sess_options=options, providers=["CPUExecutionProvider"]
        )
        # Compare the same representable inputs to isolate graph conversion error.
        # FP16 input rounding is measured separately against the original reference.
        rounded_input = raw_test.astype(dtype).astype(np.float32)
        with torch.no_grad():
            precision_reference = wrapper(torch.from_numpy(rounded_input)).numpy()
        for batch in [1, 17, len(raw_test)]:
            actual = session.run(None, {"features": raw_test[:batch].astype(dtype)})[
                0
            ].astype(np.float32)
            assert (
                actual.shape == (min(batch, len(raw_test)), N_CLASSES)
                and np.isfinite(actual).all()
            )
            np.testing.assert_allclose(
                actual.sum(1), 1, atol=2e-3 if precision == "fp16" else 2e-5
            )
            np.testing.assert_allclose(
                actual,
                precision_reference[:batch],
                rtol=2e-2 if precision == "fp16" else 3e-4,
                atol=2e-2 if precision == "fp16" else 3e-5,
                err_msg=f"{name} {precision}, batch={batch}: ONNX vs PyTorch on identical inputs",
            )
        if precision == "fp32" and name == "gmm":
            np.testing.assert_allclose(actual, gmm_prob(Xte), rtol=5e-4, atol=5e-5)
        np.savetxt(
            ONNX_DIR / f"{name}_{precision}_ort_reference.txt",
            actual[:EXPORT_ROWS],
            fmt="%.9g",
        )
        row = dict(
            model=name,
            precision=precision,
            file=path.name,
            size_kB=path.stat().st_size / 1024,
            max_abs_probability_error=float(np.abs(actual - reference).max()),
            max_abs_input_rounding_error=float(
                np.abs(precision_reference - reference).max()
            ),
            max_abs_conversion_error=float(np.abs(actual - precision_reference).max()),
            class_agreement=float(np.mean(actual.argmax(1) == reference.argmax(1))),
            balanced_accuracy=balanced_accuracy_score(yte, actual.argmax(1)),
        )
        export_rows.append(row)
        variants.append(dict(model=name, precision=precision, file=path.name))
        print(name, precision, "passed; max Δp:", row["max_abs_probability_error"])


In [ ]:
export_summary = pd.DataFrame(export_rows)
display(export_summary)
export_summary.to_csv(ONNX_DIR / "export_validation.csv", index=False)
validation_results.to_csv(ONNX_DIR / "model_validation.csv", index=False)
test_results.to_csv(ONNX_DIR / "model_test.csv", index=False)


In [ ]:
meta = dict(
    features=FEATURES,
    class_names=CLASS_NAMES,
    class_species_fPidIndex=encoder.classes_.astype(int).tolist(),
    exported_rows=EXPORT_ROWS,
    variants=variants,
    fp16_policy="FP16 input/output; Softmax/LayerNormalization/Sub/Div and GMM Mul/ReduceSum kept FP32",
    standardization="inside graph",
    mean=scaler.mean_.tolist(),
    scale=scaler.scale_.tolist(),
    sample=DATA_SOURCE,
    momentum_range=[P_MIN, P_MAX],
    gmm_components=N_COMPONENTS,
    mapping_rule=MAPPING_RULE,
    gmm_mapping=mapping.tolist(),
    mlp=MLP_CONFIG,
    transformer=TRANSFORMER_CONFIG,
    feature_recipe="p=TPC inner momentum; TPC ratio to common pion BB; alpha_track=raw alpha*sign(charge); qT unchanged",
    tpc_reference=dict(params=BB_PARAMS, mip=MIP, pion_mass=PION_MASS),
    versions=dict(
        torch=torch.__version__, onnx=onnx.__version__, onnxruntime=ort.__version__
    ),
)
(ONNX_DIR / "model_meta.json").write_text(json.dumps(meta, indent=2))
print("Export directory:", ONNX_DIR.resolve())


## 7. O2Physics GPU inference and benchmark analysis (20 minutes)
The task `Tutorials/ML/benchmarkOnnxModel.cxx` reads the local feature matrix with `--input-file` and writes predictions with `--predictions-file`. It now accepts FP32 or FP16 output matrices for that check. Set `--fp16 true` **only for the FP16-input graph**; the flag does not convert weights.

Run the generated shell commands in the prepared O2 environment: CPU/FP32 is the baseline, then choose **CUDA** for NVIDIA or **MIGraphX** for AMD. A provider failure is not a GPU result. The task checks availability and disables CPU fallback. GPU telemetry requires the CUDA/HIP runtime libraries. No AO2D input or CCDB upload is needed: this exercise applies the models to the exact exported rows. AO2D deployment later requires identical feature construction in the task.

Benchmark several batch sizes. Each feature file contains exactly the corresponding number of rows; larger files repeat the held-out rows only for timing, not to claim extra independent statistics. Each iteration reuses its batch. Unique filenames identify model, precision, provider, batch and repeat.

In [ ]:
import shlex


In [ ]:
GPU_PROVIDER = "cuda"  # change to 'migraphx' on AMD
BATCHES = [1, 128, 1024]
REPEATS = 1  # exercise: increase to 3 to inspect variation
BENCH_DIR = ONNX_DIR / "benchmarks"
BENCH_DIR.mkdir(exist_ok=True)


In [ ]:
commands = []
jobs = []
for batch in BATCHES:
    xx = raw_test[np.arange(batch) % EXPORT_ROWS]
    input_path = BENCH_DIR / f"features_b{batch}.txt"
    np.savetxt(input_path, xx, fmt="%.9g")
    for variant in variants:
        for provider in (
            ["cpu", GPU_PROVIDER] if variant["precision"] == "fp32" else [GPU_PROVIDER]
        ):
            for repeat in range(REPEATS):
                stem = f"{variant['model']}_{variant['precision']}_{provider}_b{batch}_r{repeat}"
                output = BENCH_DIR / f"{stem}.csv"
                prediction = BENCH_DIR / f"{stem}_predictions.txt"
                args = [
                    "o2-analysistutorial-benchmark-onnx-model",
                    "-b",
                    "--model-path",
                    str((ONNX_DIR / variant["file"]).resolve()),
                    "--input-file",
                    str(input_path.resolve()),
                    "--batch-size",
                    str(batch),
                    "--features",
                    str(N_FEATURES),
                    "--provider",
                    provider,
                    "--fp16",
                    str(variant["precision"] == "fp16").lower(),
                    "--gpu-id",
                    "0",
                    "--warmup",
                    "10",
                    "--iterations",
                    "100",
                    "--cpu-threads",
                    "1",
                    "--predictions-file",
                    str(prediction.resolve()),
                    "--output-csv",
                    str(output.resolve()),
                ]
                commands.append(shlex.join(args))
                jobs.append(
                    dict(
                        **variant,
                        provider=provider,
                        batch=batch,
                        repeat=repeat,
                        predictions=str(prediction),
                        csv=str(output),
                    )
                )
script = ONNX_DIR / "run_benchmarks.sh"
script.write_text(
    "#!/usr/bin/env bash\nset -euo pipefail\n" + "\n".join(commands) + "\n"
)
(ONNX_DIR / "benchmark_jobs.json").write_text(json.dumps(jobs, indent=2))
print("Run in the O2 terminal:", shlex.join(["bash", str(script.resolve())]))
print("First command:", commands[0])


### Exercise 3 — make and interpret the performance plots
1. Run CPU plus your GPU provider, then run the cells below. Missing outputs are reported as **not run**, never treated as successful checks or filled with invented measurements.
2. Compare **all probabilities** with the FP32 Python reference. Inspect FP16's maximum error and class agreement; a close decision boundary can change class. This is numerical agreement, distinct from agreement with species labels.
3. Plot throughput and latency against batch size, with separate curves for model, precision and provider. Compare CPU and GPU only at matched settings; inspect FP16 versus FP32 on the same GPU.
4. Combine throughput with the exported model's balanced accuracy. Is the fastest configuration acceptable for the desired classification quality? A small network may be faster on CPU because launch and transfer overhead dominates.

Timing includes synchronous inference, host/device transfers and output allocation. It excludes model load, text I/O, feature construction, warm-up and the additional prediction-export call. Device-wide GPU memory snapshots are not model-only peak memory. This is inference on repeated prepared batches, not end-to-end O2 analysis throughput. For repeated timings report spread as well as the mean.

In [ ]:
# Run after O2. Numerical parity thresholds are teaching tolerances, not detector requirements.
parity = []
for job in jobs:
    path = Path(job["predictions"])
    if not path.exists():
        continue
    expected = np.loadtxt(ONNX_DIR / f"{job['model']}_fp32_reference.txt", ndmin=2)
    expected = expected[np.arange(job["batch"]) % EXPORT_ROWS]
    actual = np.loadtxt(path, ndmin=2)
    assert actual.shape == expected.shape and np.isfinite(actual).all(), path
    np.testing.assert_allclose(
        actual.sum(1), 1, atol=2e-3 if job["precision"] == "fp16" else 2e-5
    )
    tol = 2e-2 if job["precision"] == "fp16" else 5e-5
    np.testing.assert_allclose(actual, expected, atol=tol, rtol=tol)
    truth = yte[:EXPORT_ROWS][np.arange(job["batch"]) % EXPORT_ROWS]
    parity.append(
        dict(
            model=job["model"],
            precision=job["precision"],
            provider=job["provider"],
            batch_size=job["batch"],
            repeat=job["repeat"],
            max_abs_error=np.abs(actual - expected).max(),
            class_agreement=np.mean(actual.argmax(1) == expected.argmax(1)),
            accuracy_on_batch=np.mean(actual.argmax(1) == truth),
        )
    )
if parity:
    parity = pd.DataFrame(parity)
    display(parity)
    parity.to_csv(BENCH_DIR / "prediction_checks.csv", index=False)
else:
    print(
        "O2 prediction checks: NOT RUN. Execute run_benchmarks.sh in the prepared O2 environment."
    )


In [ ]:
frames = []
for job in jobs:
    path = Path(job["csv"])
    if not path.exists():
        continue
    frame = pd.read_csv(path)
    required = {
        "seconds",
        "iterations",
        "batch_size",
        "features",
        "provider",
        "precision",
        "model",
        "samples_per_second",
    }
    if not required.issubset(frame.columns):
        raise ValueError(f"Unexpected CSV schema: {path}")
    assert np.isfinite(
        frame[
            ["seconds", "iterations", "batch_size", "features", "samples_per_second"]
        ].to_numpy(dtype=float)
    ).all()
    assert (
        len(frame) == 1 and (frame.seconds > 0).all() and (frame.iterations > 0).all()
    )
    assert (
        int(frame.batch_size.iloc[0]) == job["batch"]
        and int(frame.features.iloc[0]) == N_FEATURES
    )
    assert (
        frame.provider.iloc[0] == job["provider"]
        and frame.precision.iloc[0] == job["precision"]
    )
    assert Path(frame.model.iloc[0]).name == job["file"]
    frame["model_name"] = job["model"]
    frame["repeat"] = job["repeat"]
    frames.append(frame)


In [ ]:
if not frames:
    print(
        "No O2 benchmark CSVs yet. Run the commands first; no timing data are fabricated."
    )
else:
    bench = pd.concat(frames, ignore_index=True)
    bench["latency_ms"] = 1000 * bench.seconds / bench.iterations
    bench["throughput"] = bench.batch_size * bench.iterations / bench.seconds
    np.testing.assert_allclose(bench.throughput, bench.samples_per_second, rtol=1e-8)
    group = ["model_name", "provider", "precision", "batch_size"]
    summary = (
        bench.groupby(group)
        .agg(
            throughput=("throughput", "mean"),
            throughput_std=("throughput", "std"),
            latency_ms=("latency_ms", "mean"),
            latency_std=("latency_ms", "std"),
            runs=("seconds", "size"),
        )
        .reset_index()
    )
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    for key, part in summary.groupby(["model_name", "provider", "precision"]):
        part = part.sort_values("batch_size")
        label = "/".join(key)
        for ax, value, error in [
            (axes[0], "throughput", "throughput_std"),
            (axes[1], "latency_ms", "latency_std"),
        ]:
            ax.errorbar(
                part.batch_size,
                part[value],
                yerr=part[error].fillna(0),
                marker="o",
                label=label,
            )
    axes[0].set(
        xscale="log",
        yscale="log",
        xlabel="Batch size",
        ylabel="Tracks / second",
        title="Throughput",
    )
    axes[1].set(
        xscale="log",
        yscale="log",
        xlabel="Batch size",
        ylabel="ms / batch",
        title="Latency, transfers included",
    )
    axes[1].legend(fontsize=7)
    plt.tight_layout()
    plt.show()
    joined = summary.merge(
        export_summary[["model", "precision", "balanced_accuracy"]],
        left_on=["model_name", "precision"],
        right_on=["model", "precision"],
        validate="many_to_one",
    )
    fig, ax = plt.subplots(figsize=(8, 5))
    largest = joined[joined.batch_size == joined.batch_size.max()]
    for _, row in largest.iterrows():
        label = f"{row.model_name}/{row.provider}/{row.precision}"
        ax.scatter(row.throughput, row.balanced_accuracy, label=label)
    ax.set(
        xscale="log",
        xlabel="Tracks / second (largest measured batch)",
        ylabel="Export-test balanced accuracy",
        title="Quality versus inference throughput",
    )
    ax.legend(fontsize=7)
    plt.tight_layout()
    plt.show()
    display(summary)
    summary.to_csv(BENCH_DIR / "benchmark_summary.csv", index=False)
    print(
        "Accuracy axis comes from Python export checks; use prediction_checks.csv to confirm O2 numerical agreement."
    )


**Closing questions:** What changes when Gaussian components overlap? Why can seven components represent only three species? Which AP coordinates describe a mother and which depend on the daughter row? Did size or architecture improve held-out performance? Did FP16 change numerical answers enough to matter? Was the GPU faster for every batch?

References: [scikit-learn mixture models](https://scikit-learn.org/stable/modules/mixture.html), [PyTorch cross entropy](https://docs.pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html), [ONNX export](https://docs.pytorch.org/docs/stable/onnx.html), [ONNX Runtime FP16 conversion](https://onnxruntime.ai/docs/performance/model-optimizations/float16.html), [execution providers](https://onnxruntime.ai/docs/execution-providers/). AP and assigned-hypothesis TOF conventions were checked against local `PWGLF/DataModel/LFStrangenessTables.h` and `DPG/Tasks/TPC/tpcSkimsTableCreator.cxx`.